# MyKLPT2

In [20]:
def two_squares_none(n):   #CF source two_squares
    """
    Write the integer `n` as a sum of two integer squares if possible;
    otherwise raise a :exc:`ValueError`.

    INPUT:

    - ``n`` -- integer

    OUTPUT: a tuple `(a,b)` of nonnegative integers such that
    `n = a^2 + b^2` with `a <= b`.

    EXAMPLES::

        sage: two_squares(389)
        (10, 17)
        sage: two_squares(21)
        Traceback (most recent call last):
        ...
        ValueError: 21 is not a sum of 2 squares
        sage: two_squares(21^2)
        (0, 21)
        sage: a, b = two_squares(100000000000000000129); a, b                           # needs sage.libs.pari
        (4418521500, 8970878873)
        sage: a^2 + b^2                                                                 # needs sage.libs.pari
        100000000000000000129
        sage: two_squares(2^222 + 1)                                                    # needs sage.libs.pari
        (253801659504708621991421712450521, 2583712713213354898490304645018692)
        sage: two_squares(0)
        (0, 0)
        sage: two_squares(-1)
        Traceback (most recent call last):
        ...
        ValueError: -1 is not a sum of 2 squares

    TESTS::

        sage: for _ in range(100):                                                      # needs sage.libs.pari
        ....:     a = ZZ.random_element(2**16, 2**20)
        ....:     b = ZZ.random_element(2**16, 2**20)
        ....:     n = a**2 + b**2
        ....:     aa, bb = two_squares(n)
        ....:     assert aa**2 + bb**2 == n

    Tests with numpy and gmpy2 numbers::

        sage: from numpy import int16                                                   # needs numpy
        sage: two_squares(int16(389))                                                   # needs numpy
        (10, 17)
        sage: from gmpy2 import mpz
        sage: two_squares(mpz(389))
        (10, 17)

    ALGORITHM:

    See https://schorn.ch/lagrange.html
    """
    n = ZZ(n)

    if n <= 0:
        if n == 0:
            z = ZZ.zero()
            return (z, z)
        return 

    if n.nbits() <= 32:
        from sage.rings import sum_of_squares
        return sum_of_squares.two_squares_pyx(n)

    # Start by factoring n (which seems to be unavoidable)
    F = n.factor(proof=False)

    # First check whether it is possible to write n as a sum of two
    # squares: all prime powers p^e must have p = 2 or p = 1 mod 4
    # or e even.
    for p, e in F:
        if e % 2 and p % 4 == 3:
            return 

    # We run over all factors of n, write each factor p^e as
    # a sum of 2 squares and accumulate the product
    # (using multiplication in Z[I]) in a^2 + b^2.
    from sage.rings.finite_rings.integer_mod import Mod
    a = ZZ.one()
    b = ZZ.zero()
    for p, e in F:
        if e >= 2:
            m = p ** (e // 2)
            a *= m
            b *= m
        if e % 2:
            if p == 2:
                # (a + bi) *= (1 + I)
                a, b = a - b, a + b
            else:  # p = 1 mod 4
                # Find a square root of -1 mod p.
                # If y is a non-square, then y^((p-1)/4) is a square root of -1.
                y = Mod(2, p)
                while True:
                    s = y**((p - 1) / 4)
                    if not s * s + 1:
                        s = s.lift()
                        break
                    y += 1
                # Apply Cornacchia's algorithm to write p as r^2 + s^2.
                r = p
                while s * s > p:
                    r, s = s, r % s
                r %= s

                # Multiply (a + bI) by (r + sI)
                a, b = a * r - b * s, b * r + a * s

    a = a.abs()
    b = b.abs()
    assert a * a + b * b == n
    return (a, b) if a <= b else (b, a)



def trysolve(f, y):
    y = ZZ(y)
    if y <= 0:
        return
    if y.is_pseudoprime():
        return two_squares_none(y)
    else: 
        return

In [2]:
##def equivalent_prime_ideal(I):
##    Q = I.quaternion_algebra()
##    N0 = I.norm()
##    L = IntegralLattice(I.gram_matrix()).lll().basis_matrix() * I.basis_matrix()
##    bnd = 1
##    while True:
##        for _ in range(5):
##            δ = Q(sum(randrange(-bnd,bnd+1)*v for v in L))
##            N = ZZ(δ.reduced_norm() / N0)
##            if N.is_pseudoprime():
##                break
##        else:
##            bnd += 1
##            continue
##        break
##    print(f'{δ = }')
##    assert δ in I
##    J = I * (δ.conjugate() / N0)
##    del N0
##    print(f'{I = }')
##    print(f'{N = }')
##    return J

In [3]:
def norm_and_generator(I):
    N = ZZ(I.norm())
    O0 = I.left_order()
    bnd = 1
    while True:
        for _ in range(5):
            α = sum(randrange(-bnd,bnd+1)*b for b in I.basis())
            if gcd(α.reduced_norm(), N**2) == N:
                break
        else:
            bnd += 1
            continue
        break
    else:
        assert False
#    print(f'{α = }')
    assert I == O0*N + O0*α
    return N, α

In [4]:
def represent_integer(O0, rhs):
    Q = O0.quaternion_algebra()
    ii,jj,kk = Q.gens()
    if Q.quaternion_order(Q.basis()).discriminant() != 4 * ii**2 * jj**2:
        raise NotImplementedError
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF(1, 0, q)    # x^2 + y^2
    cbnd = isqrt(rhs / 2 / p)
    dbnd = isqrt(rhs / 2 / (p*q))
    if not cbnd or not dbnd:
        print('erreur lN1 trop petit')
        return
    for _ in range(999):
        c = randrange(1,cbnd+1)
        d = randrange(1,dbnd+1)
        rhs1 = rhs - p*nf(c,d)

        sol = trysolve(nf, rhs1)
        if sol is not None:
            a,b = sol
            break
    else:
        print('Pas de solution trouvée')
        return
    γ = Q([a,b,c,d])
#    print(f'{γ = }')
    assert γ in O0
    assert γ.reduced_norm() == rhs
    return γ

In [5]:
def ideal_mod_constraint(N, α, γ):
    ii,jj,kk = α.parent().gens()
    mat = matrix(GF(N), [list(elt) for elt in (γ*jj, γ*kk, α, ii*α, jj*α, kk*α)])
    ker = mat.left_kernel_matrix()
    return next(filter(bool, ker[:,:2].change_ring(ZZ)))  #TODO kernel rank > 1?

In [22]:
def strong_approximation(N, α, C, D, rhs):
    ii,jj,kk = α.parent().gens()
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF([1, 0, q])    # x^2 + y^2
    rhs1 = Mod(rhs, N) / p / nf(C,D)
    if not rhs1.is_square():
        l = next(l for l in rhs.prime_divisors() if not Mod(l,N).is_square())
        rhs //= l
        rhs1 //= l
        assert rhs1.is_square()
    λ = ZZ(rhs1.sqrt())    #lambda n'est pas le bon carré ? approx ?
    print(f'{λ = }')
    λC,λD = (λ * vector(GF(N), (C,D))).change_ring(ZZ)

    x,y,z,t = polygens(ZZ, 'x,y,z,t')
    eqn = rhs - nf(N*x,N*y) - p*nf(λC+N*z, λD+N*t)
    print(f'{eqn =}')
    print(f'{eqn(0,0,z,t) =}')
    assert eqn % N == 0
    eqn1 = eqn // N % N
    U, V, W = eqn1[z], eqn1[t], -eqn1.constant_coefficient()
    assert eqn1 == U*z + V*t - W

    # Petit-Smith
    lat = matrix([[U,0,1,0],[V,0,0,1],[-W,1,0,0]]).stack(N*identity_matrix(4))
    scal = diagonal_matrix([N**2, N, 1, 1])
    for row in matrix(ZZ, filter(bool, (lat * scal).LLL() * ~scal)):
#        print(row)
        if row[1] < 0:
            row = -row
        if not row[0] and row[1] == 1:
            sol0 = row[2:]
            break
    else:
        assert False, 'should never happen'
    import fpylll
    mat = matrix(filter(bool, matrix([[V,-U],[N,0],[0,N]]).LLL()))
    assert mat.dimensions() == (2, 2)
    lat = fpylll.IntegerMatrix(2, 2)
    for i,row in enumerate(mat):
        for j,c in enumerate(row):
            lat[i,j] = c
    gso = fpylll.GSO.Mat(lat)  #TODO lengths are slightly off when q>1
    gso.update_gso()
    cnt = 10
    seen = set()
    count_negatif = 0   #compteur ajouté
    while True:
        enum = fpylll.Enumeration(gso, cnt, fpylll.EvaluatorStrategy.BEST_N_SOLUTIONS)
        rs = enum.enumerate(0, 2, N**2, 0, tuple(mat.solve_left(sol0)))
        for r in rs:
            z,t = sol0 - vector(ZZ,r[1])*mat
            assert eqn1(0,0,z,t) % N == 0
            if (z,t) in seen:
                continue
            seen.add((z,t))
            assert eqn(0,0,z,t) % N**2 == 0
            rhs2 = ZZ(eqn(0,0,z,t)) // N**2
            print(f'{rhs2=}')
            print(rhs2.factor())
            diff = p*nf(λC+N*z, λD+N*t)
            #print(f'{diff =}')
            marge_disc = (diff / p^3).numerical_approx()
            print(f'{marge_disc =}')
            #print(f'{rhs =}')
            #print(f'{λ =}')
            print(f'{z =}', f'{t = }')
            if rhs2 <= 0:
                count_negatif = count_negatif + 1
            else:
                count_negatif = 0
            assert count_negatif < 10
            sol = trysolve(nf, rhs2)          #rhs2 somme de deux carré? positif ? 
            if sol is not None:
                x,y = sol
                break
        else:
            if len(rs) < cnt:
                raise NotImplementedError
            cnt *= 2
            continue
        break

    γ = (λC*jj + λD*kk) + N*(x + y*ii + z*jj + t*kk)
    print(f'{γ = }')
    print(f'{γ.reduced_norm() = }')
    assert γ.reduced_norm() == rhs
    return γ

In [43]:
def liste(facto):
    k = len(facto)
    facto_liste = []
    for i in range(k):
        facto_liste.append([facto[i][0], facto[i][1]])
    return facto_liste

def rand_facto(N,factoN,D,l):

    #On suppose N produit de premier distincts, listés dans factoN.
    #Le discriminant est donné positif
    
    k = len(factoN)
    facto1 = liste(factoN)
    N1 = N
    N2 = 1
        
    liste_indice = [0 .. k-1]
    while N2 <= D^3 and k>0:
        i = liste_indice[randint(0,k-1)]
        prime = facto1[i][0]
        exp = facto1[i][1]
        if exp > 0:
            N2 = N2*prime
            facto1[i][1] = exp-1
            N1 = N1 // prime
        else:
            liste_indice.remove(i)
        k = len(liste_indice)
    
    assert N == N1*N2
    if l*N1 > 3*D and N2 > D^3:   #3 pour avoir une marge d'essais pour Cornacchia
        return N1, N2    #On préfère tj N2 plus grand
    else:
        return rand_facto(N,factoN,D,l)

In [54]:
def klpt(I, N, factoN):
    print(f'{N = }')
    ii,jj,kk = I.quaternion_algebra().gens()
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF([1, 0, q])  #Peut-être jj plutôt ??
    gcdN = N.gcd(p)
    
    FactoG = gcdN.factor(proof=False)
    for p, e in FactoG:
        if e % 2 and p % 4 == 3:
            print(FactoG)
            raise ValueError('Gcd bloquant Cornacchia')
            
    while True:   #Necesaire ?

        if not ZZ(I.norm()).is_pseudoprime():
            raise NotImplementedError         #On suppose I de norme premier, quitte a faire une equivalence. 
#            I = equivalent_prime_ideal(I)

        l,α = norm_and_generator(I)   
        print(l)
        O0 = I.left_order()
        abs_disc = O0.discriminant()   #discriminant positif
            
        test = True
        while test:
            N1, N2 = rand_facto(N,factoN,abs_disc,l)   #TODO : Tester si la facto est déjà vue ?
            assert N1*N2 == N
            print('Tentative repinteger')
            print(f'{N1 = }')
            print(f'{N2 = }')
            γ = represent_integer(O0, l*N1)
            if γ is not None:
                test = False
        print('Marge erreur sur N2', (N2/(abs_disc)^3).numerical_approx())
        print('gcd N2 disc', N2.gcd(abs_disc))
        
        C,D = ideal_mod_constraint(l, α, γ)
        print(f'{C = }')
        print(f'{D = }')
        if l.divides(nf(C,D)):
            raise NotImplementedError('bad')

        µ = strong_approximation(l, α, C, D, N2)
        
        return γ * µ

# Test Torsions

## CONTEXTE : Courbe avec anneaux d'endomorphisme

In [1]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
from sage.schemes.elliptic_curves.hom_velusqrt import EllipticCurveHom_velusqrt
from sage.schemes.elliptic_curves.weierstrass_morphism import *
from sage.groups.generic import order_from_multiple
from sage.schemes.elliptic_curves.ell_curve_isogeny import compute_isogeny_bmss
from sage.schemes.elliptic_curves.hom_frobenius import EllipticCurveHom_frobenius
from sage.libs.libecm import ecmfactor
from sage.misc.search import search
import time

In [4]:
# CONTEXTE 1 :
db = HilbertClassPolynomialDatabase() #discriminant jusque -9999

def curve_from_disc(D,p):
    #on suppose D negatif
    H = db[D]
    Fp = GF(p)
    PFp.<Xp> = PolynomialRing(Fp)
    H = PFp(H)
    Hr = H.factor()[0][0]
    d = Hr.degree()
    Fd = GF(p^d)
    PFd.<Xd> = PolynomialRing(Fd)
    Hr = Hr(Xd)
    Hj = (Hr.factor()[0][0])
    j = -Hj(0)
    E = EllipticCurve_from_j(j)
    return E
    
D = -9999
p = 109
E = curve_from_disc(D,p)
j = E.j_invariant()
H = db[-D]
assert H(j) == 0
Fq = E.base_ring()
dq = Fq.degree()
q = p^dq
P = E.random_point()

f = 3 # obtenue avec -9999 = - 3^2*11*101
K.<rK> = QuadraticField(-1111)
rD = 3*rK
dK = K.discriminant()
wK = (dK + rK)/2
OK = K.maximal_order()
O = K.order([1,f*wK])
assert f == O.conductor()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*q
fm = sqrt(Dm/(K.discriminant()))



ValueError: file not found in the Kohel database

In [5]:
# CONTEXTE 2 : Exemple de Jao "small"

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ?
dK = K.discriminant()
f = O.conductor()
D = (f^2)*dK
rD = f*rK
wK = (dK + rK)/2

n = 1
Fpn = GF(p^n)
E_eval = EllipticCurve(Fpn, [79,44])
P = E_eval.random_point()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

l = 5000000029 


## ETUDE DE LA TORSION DE E

In [50]:
#Contexte choisi : exemple small Jao

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()     

def etude_torsions(E,p,O,K,degmax):
    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    rD = f*rK
    wK = (dK + rK)/2
    CE = E.cardinality_pari()
    tracef = E.trace_of_frobenius()
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/(K.discriminant()))
    s = int((-fm*dK + tracef)/2)
    s2 = (-fm*dK - tracef)/2
    frob = fm*wK + s
    frob2 = -(fm*wK + s2)
    assert (frob^2 - tracef*frob + p) == 0
    assert (frob2^2 - tracef*frob2 + p) == 0

    if dK%4 == 1 :
        a = int((tracef - fm)/2)
    else :
        a = int(tracef/2)
    Nmax = gcd(a-1,fm/f)
    torsions = [Nmax]

    torsions_candidats = []
    candidats_clapoti = []
    if Nmax^2 > -D:
        facto = Nmax.factor()
        candidats_clapoti.append([Nmax,1])
    torsions_candidats.append([Nmax,1])

    #traces = [tracef]
    #cards = [CE]
    frobd = frob
    frob2d = frob2
    q = p
    for d in [2 .. degmax]:
        q = q*p
        frobd = frobd*frob
        frob2d = frob2d*frob2
        tracefd = frobd + frob2d
        Dm = tracefd^2 - 4*q
        fm = int(sqrt(Dm/(dK)))
        assert (frobd^2 - tracefd*frobd + q) == 0
        assert (frob2d^2 - tracefd*frob2d + q) == 0
        if dK%4 == 1 :
            a = int((tracefd - fm)/2)
        else :
            a = int(tracefd/2)
        Nd = gcd(a-1,fm/f)
        torsions.append(Nd)
        if Nd^2 > (-D):
            facto = Nd.factor()
            candidats_clapoti.append([Nd,d])
        torsions_candidats.append([Nd,d])
        #traces.append(tracefd)
        #cards.append(q + 1 - tracefd)
    #print('Torsions', torsions)
    #print('Candidats clapoti', candidats_clapoti)
    #print('Torsions candidats',torsions_candidats)
    return torsions_candidats

degmax = 120
torsions = etude_torsions(E,p,O,K,degmax)
len(torsions)

120

# TEST SOLUTIONS KLPT

In [91]:
def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"





def sol_KLPT(N,factoN,aa):

    #Utiliser KLPT pour résoudre l'equation définie par N et aa
    while True :
    
        l, α = aa.gens_two()
        Quat.<i,j,k> = QuaternionAlgebra(-1, dK)
        assert (α[0] + α[1]*j).reduced_norm() == α.norm()
        assert (α[0] + α[1]*j).reduced_trace() == α.trace()
        assert j.reduced_norm() == rK.norm() and j.reduced_trace() == rK.trace()
        r = α.parent().number_field().gen()
        assert (r+1)/2 in O   #Necessaire ? ordre de discriminant = 1 mod 4 ?
        OO = Quat.quaternion_order([1, i, (1+j)/2, (i+k)/2])
        I = OO*l + OO*(α[0] + α[1]*j)
        print(f'{I = }')


        elt = klpt(I,N,factoN)
        assert elt in I
        print(f'{elt = }', '| norm:', elt.reduced_norm().factor())
    

        b = elt[0] + elt[2]*r
        c = elt[1] + elt[3]*r
        #while b and c and b/2 in aa and c/2 in aa:  # can we avoid this a priori in KLPT?
            #b /= 2
            #c /= 2
        print(f'{b = }')
        print(f'{c = }')
        assert b in aa
        assert c in aa
        bb = O.ideal([g*b.conjugate()/aa.norm() for g in aa.gens()])
        cc = O.ideal([g*c.conjugate()/aa.norm() for g in aa.gens()])
        Nb = bb.norm()
        Nc = cc.norm()
        print(f'{Nb = }')#.factor())
        print(f'{Nc = }')#.factor())
        if ZZ(Nb + Nc) == N:
            break
        else:
            print('Erreur sur Nb, Nc')
    if Nb.gcd(Nc) == 1:
        print('solution premier entre eux')
        return Nb, Nc, 1
    else :
        print('solution avec gcd>1')
        return Nb, Nc, Nb.gcd(Nc)
        
        

In [92]:
def reduction_ideal(aa):
    qa = aa.quadratic_form()
    ra = qa.reduced_form()
    lr = ra.small_prime_value()
    aar = ideal_de_norme(lr,f,D)
    if aar.is_equivalent(aa):
        return aar
    else: 
        return aar.conjugate()


In [93]:
aa = ideal_de_norme(l,f,D)
aa = reduction_ideal(aa)
aa.norm()

NameError: name 'l' is not defined

In [39]:
k_candidats = len(candidats_KLPT)
for i in [ 0 .. k_candidats - 1]:
    N = candidats_KLPT[i][0]
    factoN = candidats_KLPT[i][1]
    Nb, Nc, verif = sol_KLPT(N,factoN,aa)
    

In [40]:
aa = ideal_de_norme(l,f,D)
aa = reduction_ideal(aa)
l, α = aa.gens_two()
Quat.<i,j,k> = QuaternionAlgebra(-1, dK)
OO = Quat.quaternion_order([1, i, (1+j)/2, (i+k)/2])
I = OO*l + OO*(α[0] + α[1]*j)
Q = OO.quaternion_algebra()
ii,jj,kk = Q.gens()
if Q.quaternion_order(Q.basis()).discriminant() != 4 * ii**2 * jj**2:
    raise NotImplementedError

q, p = ZZ(-ii**2), ZZ(-jj**2)

cbnd = isqrt(l*2 / 2 / p)
dbnd = isqrt(l*2 / 2 / (p*q))
cbnd, dbnd, p, q

(0, 0, 38669866235, 1)

In [41]:
int(log(sqrt(p^7),2))

123

In [77]:
N = 2^130 + randint(1,2^10)   #Taille de N pour avoir un rhs2 positif : log(D^(3,5),2) = 52
# Exemple qui fonctionne 2^180 :N = 1532495540865888858358347027150309183618739122183602389
# Exemple qui fonctionne 2^130 :N = 1361129467683753853853498429727072846691
factoN = N.factor()
print(factoN)
print(p.factor())
sol_KLPT(N,factoN,aa)

31547 * 419999 * 1514197 * 67843888662238392103307
10000000019
I = Fractional ideal (1/2 + 1066081/2*j, 1/2*i + 1066081/2*k, 1080229*j, 1080229*k)
N = 1361129467683753853853498429727072845987
1080229
Tentative repinteger
N1 = 1514197
N2 = 898911745092450885752315207154071
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 1514197
N2 = 898911745092450885752315207154071
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 1514197
N2 = 898911745092450885752315207154071
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 = 419999
N2 = 3240792163037897361311570812614013
Pas de solution trouvée
Tentative repinteger
N1 

KeyboardInterrupt: 

In [56]:
N.gcd(p)

77

# TEST SOLUTION PEGASIS

In [51]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal

#Contexte choisi : exemple small Jao

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()   

rK = K.gens()[0]
dK = K.discriminant()
f = O.conductor()
D = (f^2)*dK
rD = f*rK
wK = (dK + rK)/2
CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))


In [52]:
def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"

In [101]:
l = 5000000029
L = ideal_de_norme(l,f,D)
L

Ideal (5127816019/2*rK + 1/2, 5000000029*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I

In [54]:
#TODO :

#def candidats_torsions_clapoti : liste des torsions ? mieux ?
#def ideal_friable (Pour calculer be, ce et leurs isogénies)
#def coin_equation
#def is_Bgood #Pour tester Pegasis4D



In [103]:
def ideal_to_element(a,L,O):
    #On suppose a equivalent à L. On cherche alpha dans L tel que a = (alphabar / N(L))L
    assert a.is_equivalent(L)
    B = a*(L.conjugate())
    alphabar = (B.gens_reduced())[0]
    alpha = alphabar.conjugate()
    assert NumberFieldOrderIdeal(O,alphabar) == B
    assert alpha in L
    return alpha

def element_to_ideal(alpha,L,O):
    #On suppose alpha dans L et on lui asocie un idéal equivalent
    assert alpha in L
    alphabar = alpha.conjugate()
    B = NumberFieldOrderIdeal(O,alphabar)
    a2 = B*L
    g1, g2 = a2.gens_reduced()
    a = NumberFieldOrderIdeal(O,[g1/(L.norm()), g2/(L.norm())])
    assert a.is_equivalent(L)
    return a

qL = L.quadratic_form()
rL = qL.reduced_form()
a = NumberFieldOrderIdeal(O,[rL[0], (-rL[1] + f*rK)/2])
rL, a.quadratic_form()   #TODO WTF pas la même pour norme 23 ?? Probleme important ?

(91759*x^2 - 48971*x*y + 111891*y^2, 91759*x^2 - 48971*x*y + 111891*y^2)

In [104]:
a == NumberFieldOrderIdeal(O,a.quadratic_form())

True

In [105]:
alpha = ideal_to_element(a,L,O)
a2 = element_to_ideal(alpha,L,O)
a == a2

True

In [106]:
def base_vecteurs_courts(L,O):
    qL = L.quadratic_form()
    rL = qL.reduced_form()
    a = NumberFieldOrderIdeal(O,[rL[0], (-rL[1] + f*rK)/2])
    b = NumberFieldOrderIdeal(O,[rL[2], (rL[1] + f*rK)/2])
    alpha = ideal_to_element(a,L,O)
    beta = ideal_to_element(b,L,O)
    assert NumberFieldOrderIdeal(O,[alpha,beta]) == L  #TODO possiblement faux ?
    return alpha, beta

base_vecteurs_courts(L,O)

(159/2*rK + 29284247/2, 101*rK - 12844598)

In [107]:
#Remarque : on peut réduire L avant de calculer la base de vecteurs courts, i.e qL = rL et a = L

base_vecteurs_courts(a,O)

(91759, 1/2*rK + 48971/2)

In [123]:
def liste_premiers_splits(D,borne_B):
    liste_B = []
    i = 2
    while len(liste_B) < borne_B :
        if kronecker(D,i) == 1:
            liste_B.append(i)
        i = next_prime(i)
    return liste_B

liste_B = liste_premiers_splits(D,9)  #assurer une taille minimale de la liste ?
liste_B

[3, 19, 23, 31, 43, 61, 67, 71, 89]

In [124]:
def make_liste_ideq(L,O,m,liste_B,CE):
    # présence de CE : pour evaluer des endomorphismes, on veut des normes premieres avec le cardinal
    # On suppose que l'on ne veut evaluer l'isogenie que sur des points du corps de base Fq
    alpha, beta = base_vecteurs_courts(L,O)
    liste_ideq = []
    for x in [0 .. m]:   #on evite de creer gamma et -gamma
        for y in [-m .. m]:
            if (x != 0 or y > 0):   #on evite les cas x = 0 et y <= 0
                gamma = x*alpha + y*beta
                if gamma == 0:
                    raise ValueError('erreur base courte liée')
                I = element_to_ideal(gamma,L,O)
                NI = I.norm()
                if NI.gcd(CE) != 1:
                    continue
                Nk = NI
                Ne = 1
                Ne_facto = []
                for p in liste_B:
                    exp = 0
                    while Nk%p == 0:
                        exp = exp + 1
                        Nk = Nk/p
                        Ne = Ne*p
                    if exp > 0:
                        Ne_facto.append([p,exp])
                assert NI == Nk*Ne
                liste_ideq.append([I,Nk,Ne,Ne_facto])
    return liste_ideq

liste_ideq = make_liste_ideq(a,O,2,liste_B,CE)
[b_prep[2] for b_prep in liste_ideq]

[19, 89, 1, 19, 89, 1]

In [110]:
576869.gcd(CE)

1

In [111]:
def coin_equation(N1,N2,N):
    #On veut résoudre uN1 + vN2 == N dans NN, avec uN1 gcd vN2 == 1 et N divise Nmax
    #assert N >= N1*N2  necessaire ?
    d,u0,v0 = xgcd(N1,N2)
    assert d == 1
    
    if u0 > 0:
        u, v, N, sol = coin_equation(N2,N1,N)
        return v, u, N, sol

    #On suppose u0 <= 0
        
    ku = int(((-u0)*N) // N2) + 1  #erreur de int, etrange ?
    kv = int((v0*N)//N1)
    u = u0*N + ku*N2
    v = v0*N - ku*N1
    assert u*N1 + v*N2 == N

    sol = False
    if ku > kv :
        #print('echec equation dans NN')
        return u, v, N, sol
        
    nb_sols = kv - ku + 1

    while sol == False and nb_sols >= 0:
        duv = u.gcd(v)
        if N%duv != 0:
            continue
        if ((u/duv)*N1).gcd((v/duv)*N2) == 1:
            u = u/duv
            v = v/duv
            N = N/duv
            break
        u = u + N2
        v = v - N1
        nb_sols = nb_sols - 1

    assert u*N1 + v*N2 == N
    if (u*N1).gcd(v*N2) == 1 and v > 0 and u > 0:
        sol = True
            
    return u, v, N, sol


def clapoti_equation(torsions,liste_ideq):
    k_id = len(liste_ideq)
    sols = []
    sols_ext = []
    for i in [0 .. k_id-1]:
        b_prep = liste_ideq[i]
        b, N1, M1, factoM1 = b_prep
        for j in [i .. k_id-1]:
            c_prep = liste_ideq[j]
            c, N2, M2, factoM2 = c_prep
            if N1.gcd(N2) != 1:
                #print(N1,N2,'echec N1 N2 pas premier entre eux')
                continue

            for Nmax, ext in torsions :
                sol_ext = False
                #print('Nmax =', Nmax)
                
                Gcd1 = Nmax.gcd(M1)
                Nmax_loc = Nmax/Gcd1
                Gcd2 = Nmax_loc.gcd(M2)
                Nmax_loc = Nmax_loc/Gcd2
                #print('Nmax_loc =', Nmax_loc)
                
                Nmin = N1+N2   #N1*N2 necessaire ?
                if Nmin > Nmax_loc :
                    #print(Nmin, 'echec Nmin')
                    continue
                
                #print('Tentative N =',Nmax_loc, 'N1 =', N1, 'N2 =',N2)
                u,v,N,sol = coin_equation(N1,N2,Nmax_loc)
                if sol:
                    #print('solution :',u,'*',N1,'+',v,'*',N2,'=',N)
                    sols.append([u,i,v,j,N])
                    assert u*N1+v*N2 == N
                    sol_ext = True
                #else:
                    #print(N,N1,N2,'echec equation')
                if sol_ext == True and ext not in sols_ext:
                    sols_ext.append(ext)

    return sols, sols_ext

In [112]:
coin_equation(2,3,31)

(2, 9, 31, True)

In [113]:
sols, sols_ext = clapoti_equation(torsions,liste_ideq)

In [114]:
len(sols)

9

In [115]:
sols_ext.sort()
sols_ext

[60, 66, 120]

In [26]:
(354675960/(-D)).numerical_approx()

0.00917189518692939

In [27]:
torsions_59 = [[354675960, 59]]
sols, sols_ext = clapoti_equation(torsions_59,liste_ideq)

In [70]:
sols[2]

[1, 1, 2, 1, 3]

In [72]:
liste_ideq[1]

[Ideal (45/2*rK + 1/2, 23*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I,
 1,
 23,
 [[23, 1]]]

In [73]:
CE.gcd(23)

1

In [74]:
xgcd(23,576577)

(1, 175480, -7)

In [131]:
p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()   

def test_solutions_clapoti(deg_max,prime_max,nb_id_max,l,E,K,O):
    #deg-max : degré d'extension maximal regardé
    #prime_max : nombre maximal de nombre premier split autorisé
    #nb_id_max : interval sur lequel on cherche des idéaux équivalent

    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    CE = E.cardinality_pari()
    
    L = ideal_de_norme(l,f,D)
    
    resultat = [[]]
    i = 0
    torsions_candidats = etude_torsions(E,p,O,K,deg_max)
    
    for torsions in torsions_candidats:
        resultat[i].append(torsions)
        for Bp in [1 .. prime_max]:
            liste_B = liste_premiers_splits(D,Bp)
            liste_ideq = make_liste_ideq(L,O,nb_id_max,liste_B,CE)
            sols, sols_ext = clapoti_equation([torsions],liste_ideq)
            resultat[i].append(sols)
        i = i + 1
        resultat.append([])
    return resultat

In [132]:
l = 5000000029
resultat = test_solutions_clapoti(120,9,3,l,E,K,O)


In [133]:
resultat

[[[1, 1], [], [], [], [], [], [], [], [], []],
 [[3, 2], [], [], [], [], [], [], [], [], []],
 [[1042, 3], [], [], [], [], [], [], [], [], []],
 [[3, 4], [], [], [], [], [], [], [], [], []],
 [[1, 5], [], [], [], [], [], [], [], [], []],
 [[18756, 6], [], [], [], [], [], [], [], [], []],
 [[1, 7], [], [], [], [], [], [], [], [], []],
 [[3, 8], [], [], [], [], [], [], [], [], []],
 [[1042, 9], [], [], [], [], [], [], [], [], []],
 [[3, 10], [], [], [], [], [], [], [], [], []],
 [[3013, 11], [], [], [], [], [], [], [], [], []],
 [[37512, 12], [], [], [], [], [], [], [], [], []],
 [[1, 13], [], [], [], [], [], [], [], [], []],
 [[129, 14], [], [], [], [], [], [], [], [], []],
 [[1042, 15], [], [], [], [], [], [], [], [], []],
 [[3, 16], [], [], [], [], [], [], [], [], []],
 [[1, 17], [], [], [], [], [], [], [], [], []],
 [[1069092, 18], [], [], [], [], [], [], [], [], []],
 [[1, 19], [], [], [], [], [], [], [], [], []],
 [[15, 20], [], [], [], [], [], [], [], [], []],
 [[1042, 21], [], []